# TASK 2: Medical Fine-tuning using Unsloth (QLoRA)
# Platform: Google Colab (T4 GPU)

In [1]:
# CELL 1: Install Required Libraries
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-9k54x9pq/unsloth_8afa2f26860c4393a6628f8045359a12
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-9k54x9pq/unsloth_8afa2f26860c4393a6628f8045359a12
  Resolved https://github.com/unslothai/unsloth.git to commit 7ef8cde3c22a3bc2240353353caa21051a42ea90
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.2 MB/s eta 0:00:00


In [2]:
# ──────────────────────────────────────────
# CELL 2: Import Libraries
# ──────────────────────────────────────────

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

print("✅ Libraries imported successfully!")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Libraries imported successfully!
GPU available: True
GPU name: Tesla T4


In [4]:
# ──────────────────────────────────────────
# CELL 3: Load Base Model (4-bit Quantized)
# ──────────────────────────────────────────

# Settings
max_seq_length = 2048   # How long each text can be
dtype = None            # Auto-detect best data type
load_in_4bit = True     # Use 4-bit quantization = less memory needed

# Load Llama 3 model with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",  # Base model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("✅ Model loaded successfully!")
print(f"Model has {sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters")

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


✅ Model loaded successfully!
Model has 4541M parameters


In [5]:
# ──────────────────────────────────────────
# CELL 4: Add LoRA Adapters (for fine-tuning)
# ──────────────────────────────────────────

"""
LoRA = Low-Rank Adaptation
Instead of training ALL weights, we only train small adapter layers.
This uses 95% less memory!
"""

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                        # LoRA rank - higher = more learning power
    target_modules = [
        "q_proj", "k_proj",        # Query and Key attention layers
        "v_proj", "o_proj",        # Value and Output layers
        "gate_proj", "up_proj",    # Feed-forward layers
        "down_proj"
    ],
    lora_alpha = 16,               # Scaling factor
    lora_dropout = 0,              # No dropout for Unsloth
    bias = "none",                 # No bias for speed
    use_gradient_checkpointing = "unsloth",  # Save memory
    random_state = 42,             # For reproducibility
)

print("✅ LoRA adapters added!")
print("Trainable parameters:")
model.print_trainable_parameters()

Unsloth 2026.4.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ LoRA adapters added!
Trainable parameters:
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [6]:
# ──────────────────────────────────────────
# CELL 5: Load Medical Dataset
# ──────────────────────────────────────────

"""
We use the 'medalpaca' medical Q&A dataset.
It contains thousands of medical questions and answers.
"""

# Load dataset from HuggingFace
dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train")

# Look at one example
print("Sample data:")
print(dataset[0])
print(f"\nTotal examples: {len(dataset)}")


Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

Sample data:
{'input': 'What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?', 'output': 'Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.', 'instruction': 'Answer this question truthfully'}

Total examples: 33955


In [7]:
# ──────────────────────────────────────────
# CELL 6: Format Dataset for Training
# ──────────────────────────────────────────

"""
We need to convert the raw data into a format the model understands.
Format: <instruction> [question] </instruction> <response> [answer] </response>
"""

# Define the prompt template
prompt_template = """Below is a medical question. Answer it accurately.

### Question:
{}

### Answer:
{}"""

# End-of-sequence token
EOS_TOKEN = tokenizer.eos_token  # Tells the model when to stop generating

def format_examples(examples):
    """
    Convert raw dataset examples to the training format.
    examples: a batch of data from the dataset
    """
    inputs = []

    for instruction, output in zip(examples["input"], examples["output"]):
        # Create formatted text with question and answer
        text = prompt_template.format(instruction, output) + EOS_TOKEN
        inputs.append(text)

    return {"text": inputs}

# Apply formatting to entire dataset
dataset = dataset.map(format_examples, batched=True)

print("✅ Dataset formatted!")
print("\nFormatted example:")
print(dataset[0]["text"][:500])  # Show first 500 characters

Map:   0%|          | 0/33955 [00:00<?, ? examples/s]

✅ Dataset formatted!

Formatted example:
Below is a medical question. Answer it accurately.

### Question:
What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?

### Answer:
Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.<|eot_id|>


In [8]:
# ──────────────────────────────────────────
# CELL 7: Set Up Training
# ──────────────────────────────────────────

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",    # Column name that has the text
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,           # Use 2 CPU cores for data loading

    args = TrainingArguments(
        per_device_train_batch_size = 2,    # Process 2 examples at a time
        gradient_accumulation_steps = 4,    # Simulate batch size of 8
        warmup_steps = 5,                   # Slow learning rate start
        max_steps = 60,                     # Total training steps (60 = ~5 mins)
        learning_rate = 2e-4,               # How fast to learn
        fp16 = not torch.cuda.is_bf16_supported(),   # Use fp16 if bf16 not available
        bf16 = torch.cuda.is_bf16_supported(),        # Use bf16 if available
        logging_steps = 10,                  # Print loss every 10 steps
        optim = "adamw_8bit",               # Memory-efficient optimizer
        weight_decay = 0.01,                # Regularization
        lr_scheduler_type = "linear",       # How to change learning rate
        seed = 42,                          # Reproducibility
        output_dir = "/content/medical_model",  # Where to save checkpoints
    ),
)

print("✅ Trainer is ready!")


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/33955 [00:00<?, ? examples/s]

✅ Trainer is ready!


In [9]:
# ──────────────────────────────────────────
# CELL 8: Start Training!
# ──────────────────────────────────────────

print("🚀 Starting training...")
print("This will take about 5-10 minutes on a T4 GPU")

trainer_stats = trainer.train()

print("\n✅ Training complete!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.1f} seconds")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")


🚀 Starting training...
This will take about 5-10 minutes on a T4 GPU


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 33,955 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.385434
20,0.973301
30,0.918620
40,0.881413
50,0.837715
60,0.868678



✅ Training complete!
Training time: 231.4 seconds
Final loss: 0.9775


In [10]:
# ──────────────────────────────────────────
# CELL 9: Test the Fine-tuned Model
# ──────────────────────────────────────────

# Switch model to fast inference mode
FastLanguageModel.for_inference(model)

# Test with a medical question
test_question = "What are the symptoms of diabetes mellitus type 2?"

# Prepare input
inputs = tokenizer(
    [prompt_template.format(test_question, "")],  # Empty answer = model fills it in
    return_tensors = "pt"
).to("cuda")

# Generate answer
outputs = model.generate(
    **inputs,
    max_new_tokens = 200,      # Max length of answer
    use_cache = True,
    temperature = 0.7,         # Creativity (0=robotic, 1=creative)
    do_sample = True,
)

# Decode and print the response
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Question:", test_question)
print("\nAnswer:")
print(response.split("### Answer:")[-1].strip())


Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

Question: What are the symptoms of diabetes mellitus type 2?

Answer:
Diabetes mellitus type 2 is a metabolic disorder characterized by hyperglycemia (high blood sugar) and insulin resistance. The symptoms of diabetes mellitus type 2 can include increased thirst and urination, fatigue, blurred vision, and slow healing of wounds. These symptoms can be non-specific, and therefore, the diagnosis of diabetes mellitus type 2 is often made through a combination of physical examination, medical history, and laboratory tests. Treatment for diabetes mellitus type 2 typically involves lifestyle changes, such as diet and exercise, as well as medications to control blood sugar levels.


In [11]:
# ──────────────────────────────────────────
# CELL 10: Save the Fine-tuned Model
# ──────────────────────────────────────────

# Save only the LoRA weights (small file, ~50MB)
model.save_pretrained("/content/medical_lora_model")
tokenizer.save_pretrained("/content/medical_lora_model")

print("✅ Model saved to /content/medical_lora_model")

# Optional: Save in merged format (larger but easier to use)
# model.save_pretrained_merged("/content/medical_merged", tokenizer, save_method="merged_16bit")
# print("✅ Merged model saved!")

✅ Model saved to /content/medical_lora_model


In [12]:
# ──────────────────────────────────────────
# CELL 11: More Test Questions
# ──────────────────────────────────────────

# Test multiple medical questions
test_questions = [
    "What is hypertension and how is it treated?",
    "What are the side effects of aspirin?",
    "Explain what a myocardial infarction is.",
]

for question in test_questions:
    print(f"\n{'='*50}")
    print(f"Q: {question}")

    inputs = tokenizer(
        [prompt_template.format(question, "")],
        return_tensors = "pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 150,
        use_cache = True,
        temperature = 0.7,
        do_sample = True,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split("### Answer:")[-1].strip()
    print(f"A: {answer[:300]}...")  # Show first 300 chars

Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What is hypertension and how is it treated?


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Hypertension is a condition where the blood pressure in the arteries becomes too high. It is treated with lifestyle changes, such as diet and exercise, and medications such as diuretics and beta blockers....

Q: What are the side effects of aspirin?


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: The side effects of aspirin include headache, nausea, and stomach pain....

Q: Explain what a myocardial infarction is.
A: Myocardial infarction is a medical term that refers to a heart attack, which is a serious condition that occurs when the blood flow to the heart is interrupted or blocked, resulting in damage to the heart muscle....
